# 使用推理生成例程

在开发客户服务解决方案时，最初的步骤之一是将知识库文章转换为一组 LLM 能够理解并遵循的例程（routine）。在这里，例程指的是专门为 LLM 高效执行而设计的一组分步指令。每个例程都经过精心构建，使每一步都对应一个明确的动作。动作可以包括回复用户、触发函数调用或检索额外的相关知识。

大多数内部知识库文章都很复杂，并且是为人类理解而组织的。它们通常包含错综复杂的图表、多步骤流程和决策树，这给基于 LLM 的解决方案进行有意义的推理带来了挑战。通过将这些文档拆解为例程，每条指令都可以被简化并格式化，从而引导 LLM 完成一系列小而可控的任务。这种细粒度的方法减少了歧义，使 LLM 能够有条不紊地处理信息，并降低产生幻觉或偏离预期路径的风险。

将这些知识库文章转换为例程可能既耗时又具有挑战性，尤其是对于试图构建自动化流水线的公司而言。每个例程都必须考虑各种用户场景，并清晰地定义需要执行的动作。例如，当需要函数调用时，例程必须明确指定要检索的确切信息或要执行的动作——无论是触发某个 API、检索外部数据，还是引入额外的上下文。虽然使用传统的 GPT 类模型自动化这一过程可以显著减少人工工作量，但往往会带来新的挑战。其中一些挑战包括：设计足够具体、让 LLM 能够始终如一地遵循的稳健指令；捕捉客户交互中可能出现的独特边缘情况；提供高质量的 few-shot 示例来引导模型行为；以及在某些情况下，对模型进行微调以获得更可靠或更专业的结果。

o1 已经展示出以 zero-shot 方式高效拆解这些文章并将其转换为例程集合的能力，这意味着 LLM 无需大量示例或针对类似任务的预先训练，就能理解并遵循这些指令。这最大限度地减少了所需的提示工作，因为例程结构本身就为 LLM 完成每一步提供了必要的指引。通过将任务拆解为具体动作并在需要时集成函数调用，o1 的方法确保即使是复杂的工作流也能被 LLM 顺畅处理，从而带来更高效、更具可扩展性的客户服务解决方案。

## 选择知识库文章

在本示例中，我们将使用 OpenAI 网站上一组公开的帮助中心文章，并将其转换为 LLM 可以执行的内部例程。除了将政策转换为例程之外，我们还会让模型生成函数，使 LLM 能够代表用户执行操作。这是必要的，这样 LLM 才能执行与人工客服相同的操作，并访问仅凭政策文档可能无法直接获得的额外信息。

我们将首先使用以下帮助中心文章来转换为例程：
- [How do I delete my payment method](https://help.openai.com/en/articles/8156070-how-do-i-delete-my-payment-method)
- [How can I get a Business Associate Agreement (BAA) with OpenAI?](https://help.openai.com/en/articles/8660679-how-can-i-get-a-business-associate-agreement-baa-with-openai)
- [How can I set up prepaid billing?](https://help.openai.com/en/articles/8264644-how-can-i-set-up-prepaid-billing)
- [How do I submit a VAT exemption request](https://help.openai.com/en/articles/7232908-how-do-i-submit-a-vat-exemption-request)

In [1]:
from openai import OpenAI
from IPython.display import display, HTML
import pandas as pd
from concurrent.futures import ThreadPoolExecutor
import csv

client = OpenAI()
MODEL = 'o1-preview'

我们的文章存储在一个可访问的 csv 文件中。我们会将这些文章并行传给 o1-preview，生成初始例程。

我们将政策转换为例程的指令包括：
- 将政策从面向外部的文档转换为内部 SOP 例程
- 将政策拆解为具体的动作和子动作
- 列出在步骤之间转换的具体条件
- 确定哪些地方可能需要外部知识/动作，并定义可用于获取这些信息的函数

In [2]:
articles = []

with open('../data/helpcenter_articles.csv', mode='r', encoding='utf-8') as file:
    reader = csv.DictReader(file)
    for row in reader:
        articles.append({
            "policy": row["policy"],
            "content": row["content"]
        })

In [3]:
CONVERSION_PROMPT = """
You are a helpful assistant tasked with taking an external facing help center article and converting it into a internal-facing programmatically executable routine optimized for an LLM. 
The LLM using this routine will be tasked with reading the policy, answering incoming questions from customers, and helping drive the case toward resolution.

Please follow these instructions:
1. **Review the customer service policy carefully** to ensure every step is accounted for. It is crucial not to skip any steps or policies.
2. **Organize the instructions into a logical, step-by-step order**, using the specified format.
3. **Use the following format**:
   - **Main actions are numbered** (e.g., 1, 2, 3).
   - **Sub-actions are lettered** under their relevant main actions (e.g., 1a, 1b).
      **Sub-actions should start on new lines**
   - **Specify conditions using clear 'if...then...else' statements** (e.g., 'If the product was purchased within 30 days, then...').
   - **For instructions that require more information from the customer**, provide polite and professional prompts to ask for additional information.
   - **For actions that require data from external systems**, write a step to call a function using backticks for the function name (e.g., `call the check_delivery_date function`).
      - **If a step requires the customer service agent to take an action** (e.g., process a refund), generate a function call for this action (e.g., `call the process_refund function`).
      - **Define any new functions** by providing a brief description of their purpose and required parameters.
   - **If there is an action an assistant can performon behalf of the user**, include a function call for this action (e.g., `call the change_email_address function`), and ensure the function is defined with its purpose and required parameters.
      - This action may not be explicitly defined in the help center article, but can be done to help the user resolve their inquiry faster
   - **The step prior to case resolution should always be to ask if there is anything more you can assist with**.
   - **End with a final action for case resolution**: calling the `case_resolution` function should always be the final step.
4. **Ensure compliance** by making sure all steps adhere to company policies, privacy regulations, and legal requirements.
5. **Handle exceptions or escalations** by specifying steps for scenarios that fall outside the standard policy.

**Important**: If at any point you are uncertain, respond with "I don't know."

Please convert the customer service policy into the formatted routine, ensuring it is easy to follow and execute programmatically.

"""

In [4]:
def generate_routine(policy):
    try:
        messages = [
            {
                "role": "user",
                "content": f"""
                    {CONVERSION_PROMPT}

                    POLICY:
                    {policy}
                """
            }
        ]

        response = client.chat.completions.create(
            model=MODEL,
            messages=messages
        )
        

        return response.choices[0].message.content 
    except Exception as e:
        print(f"An error occurred: {e}")

In [5]:
def process_article(article):
    routine = generate_routine(article['content'])
    return {"policy": article['policy'], "content": article['content'], "routine": routine}


with ThreadPoolExecutor() as executor:
    results = list(executor.map(process_article, articles))

我们会将例程结果存储在 dataframe 中并打印出来，以便初步查看。

In [6]:
df = pd.DataFrame(results)

# Set display options to show all text in the dataframe cells
pd.set_option('display.max_colwidth', None)

# Function to display formatted text in HTML
def display_formatted_dataframe(df):
    def format_text(text):
        return text.replace('\n', '<br>')

    df_formatted = df.copy()
    df_formatted['content'] = df_formatted['content'].apply(format_text)
    df_formatted['routine'] = df_formatted['routine'].apply(format_text)
    
    display(HTML(df_formatted.to_html(escape=False, justify='left')))

display_formatted_dataframe(df)

## 结果

审阅生成的例程后，我们可以得出以下几点洞见：
- 示例回复：模型能够有效生成 LLM 在执行政策时可以使用的示例回复（例如，“Instruct the user: ‘Confirm and purchase your initial amount of credits.’”）。
- 离散步骤：模型擅长将问题分解为 LLM 需要执行的离散动作。每条指令都定义清晰、易于理解。
- 函数定义：例程的输出包含定义清晰的函数，用于检索外部信息或触发动作（例如 `review_and_apply_tax_exemption`、`get_billing_plan`、`update_payment_method`）。
    - 这对任何成功的例程都至关重要，因为 LLM 经常需要与外部系统交互。利用函数调用是与这些系统交互并执行动作的有效方式。
- IFTTT 逻辑：模型有效地运用了 IFTTT（If This, Then That）逻辑，这对 LLM 来说非常理想（例如，“If the customer requests assistance, proceed to step 3f.”）。
    - 当原始知识库文章包含复杂的工作流和图表时，这种转换就变得极其有价值。这种复杂性可能连人类都不易理解，对 LLM 来说就更难了。IFTTT 逻辑易于理解，非常适合客户服务解决方案

## 下一步该怎么做？

现在可以将这些例程集成到智能体系统中，以解决特定的客户问题。当客户请求协助完成诸如设置预付费账单之类的任务时，我们可以使用分类器来确定应检索的合适例程，并将其提供给 LLM，以便直接与客户交互。除了向用户提供*如何*设置账单的说明之外，系统还可以代表用户执行该操作。

在将这些例程部署到生产环境之前，我们应该开发全面的评估，以测试和验证模型回复的质量。这一过程可能需要调整例程，以确保合规性和有效性。